In [0]:
%run "../common/common_logging"

In [0]:
%run "../common/common_http_client"

In [0]:
%run "../common/common_hashing"

In [0]:
%run "../common/common_config_loader"

In [0]:
%run "./ingest_manifest_writer"

In [0]:
# Get configuration from common config loader
config = get_config()
logger = get_logger(__name__)

# Arbeitnow API Configuration
ARBEITNOW_API = "https://www.arbeitnow.com/api/job-board-api"
SOURCE_NAME = "arbeitnow"

# Table names from config
BRONZE_JOB_SNAPSHOT = config.get_bronze_table("bronze_job_snapshot")

# Arbeitnow-specific field mapping
ARBEITNOW_FIELDS = {
    'job_id': 'slug',           # string - unique job identifier
    'company': 'company_name',   # string
    'title': 'title',            # string
    'description': 'description', # string
    'remote': 'remote',          # bool
    'url': 'url',                # string
    'tags': 'tags',              # array/object
    'job_types': 'job_types',    # array/object
    'location': 'location',      # string
    'created_at': 'created_at'   # int64 - Unix timestamp
}

In [0]:
def extract_arbeitnow_job_id(job):
    """Extract job ID from Arbeitnow record"""
    return job.get('slug', '')

def validate_arbeitnow_record(job):
    """Validate Arbeitnow job record structure"""
    required_fields = ['slug', 'title', 'company_name']
    for field in required_fields:
        if not job.get(field):
            return False, f"Missing required field: {field}"
    
    # Validate data types
    if not isinstance(job.get('created_at'), int):
        return False, "created_at must be int64 (Unix timestamp)"
    
    if not isinstance(job.get('remote'), bool):
        return False, "remote must be boolean"
    
    return True, None

def fetch_arbeitnow_jobs():
    """Fetch jobs from Arbeitnow API using common HTTP client"""
    try:
        # Use common HTTP client with retry logic
        http_config = HTTPClientConfig(
            max_retries=3,
            connect_timeout=10,
            read_timeout=30
        )
        client = HTTPClient(
            base_url=ARBEITNOW_API,
            config=http_config
        )
        
        logger.info(f"Fetching jobs from {ARBEITNOW_API}")
        data = client.get("")  # Empty endpoint since base_url is the full API URL
        jobs = data.get('data', [])
        
        # Validate records
        validated_jobs = []
        for job in jobs:
            is_valid, error = validate_arbeitnow_record(job)
            if is_valid:
                validated_jobs.append(job)
            else:
                logger.warning(f"Skipping invalid record - {error}")
        
        logger.info(f"Successfully fetched {len(validated_jobs)} valid jobs")
        return validated_jobs, None
        
    except Exception as e:
        logger.error(f"Failed to fetch Arbeitnow jobs: {str(e)}")
        return None, str(e)

In [0]:
import uuid
import json
import time
from datetime import datetime, timezone
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType, DateType

def generate_batch_id():
    """Generate unique batch ID for ingestion run"""
    return str(uuid.uuid4())

def generate_snapshot_id(source_name, source_job_id, batch_id):
    """Generate unique snapshot identifier"""
    return f"{source_name}_{source_job_id}_{batch_id}"

def prepare_bronze_snapshots(jobs_data, source_name, batch_id, extract_job_id_func):
    """Prepare individual job snapshots for Bronze layer"""
    if not jobs_data:
        return []
    
    bronze_records = []
    ingestion_timestamp = datetime.now(timezone.utc)
    ingestion_date = ingestion_timestamp.date()
    
    for job in jobs_data:
        source_job_id = extract_job_id_func(job)
        payload_json = json.dumps(job)
        payload_hash = calculate_payload_hash(job)  # Using common hashing
        snapshot_id = generate_snapshot_id(source_name, source_job_id, batch_id)
        
        bronze_records.append({
            'snapshot_id': snapshot_id,
            'source_name': source_name,
            'source_job_id': source_job_id,
            'batch_id': batch_id,
            'page_number': None,
            'page_size': None,
            'payload_json': payload_json,
            'payload_hash': payload_hash,
            'ingestion_timestamp': ingestion_timestamp,
            'ingestion_date': ingestion_date,
            'source_status_code': 200,
            'source_etag': None
        })
    
    return bronze_records

def ingest_to_bronze(source_name, fetch_function, api_url, extract_job_id_func,
                     log_api_response_func, start_pipeline_run_func, 
                     complete_pipeline_run_func, log_audit_func):
    """Main ingestion function - fetches API data and stores in Bronze layer
    
    Returns:
        Tuple of (success: bool, batch_id: str, records_count: int)
    """
    pipeline_name = f"bronze_ingestion_{source_name}"
    batch_id = generate_batch_id()
    start_time = time.time()
    
    # Start pipeline run in metadata
    run_control_sk = start_pipeline_run_func(pipeline_name, source_name, batch_id)
    logger.info(f"Pipeline: {pipeline_name}")
    logger.info(f"Batch ID: {batch_id}")
    
    try:
        # Fetch data from API
        logger.info(f"[1/3] Fetching data from {source_name}...")
        api_start = time.time()
        jobs_data, error = fetch_function()
        api_duration_ms = int((time.time() - api_start) * 1000)
        
        if error:
            # Log failed API response
            log_api_response_func(source_name, batch_id, api_url, 500, 
                           response_time_ms=api_duration_ms)
            
            duration = time.time() - start_time
            complete_pipeline_run_func(batch_id, 'FAILED')
            log_audit_func(batch_id, pipeline_name, 'FAILED', 
                          runtime_seconds=duration, error_message=error)
            logger.error(f"FAILED to fetch: {error}")
            return False, batch_id, 0
        
        records_fetched = len(jobs_data) if jobs_data else 0
        logger.info(f"Fetched {records_fetched} jobs in {api_duration_ms}ms")
        
        # Log successful API response
        log_api_response_func(source_name, batch_id, api_url, 200, 
                       response_time_ms=api_duration_ms)
        
        if not jobs_data:
            duration = time.time() - start_time
            complete_pipeline_run_func(batch_id, 'SUCCESS')
            log_audit_func(batch_id, pipeline_name, 'SUCCESS',
                          rows_read=0, rows_written=0, runtime_seconds=duration)
            logger.warning("No data returned")
            return True, batch_id, 0
        
        # Prepare Bronze job snapshots
        logger.info("[2/3] Processing to Bronze snapshots...")
        bronze_records = prepare_bronze_snapshots(jobs_data, source_name, batch_id, extract_job_id_func)
        records_processed = len(bronze_records)
        logger.info(f"Processed {records_processed} snapshots")
        
        # Write to Bronze layer
        logger.info(f"[3/3] Writing to {BRONZE_JOB_SNAPSHOT}...")
        bronze_schema = StructType([
            StructField("snapshot_id", StringType(), False),
            StructField("source_name", StringType(), False),
            StructField("source_job_id", StringType(), True),
            StructField("batch_id", StringType(), False),
            StructField("page_number", IntegerType(), True),
            StructField("page_size", IntegerType(), True),
            StructField("payload_json", StringType(), False),
            StructField("payload_hash", StringType(), False),
            StructField("ingestion_timestamp", TimestampType(), False),
            StructField("ingestion_date", DateType(), False),
            StructField("source_status_code", IntegerType(), True),
            StructField("source_etag", StringType(), True)
        ])
        
        df = spark.createDataFrame(bronze_records, schema=bronze_schema)
        df.write.mode('append').saveAsTable(BRONZE_JOB_SNAPSHOT)
        logger.info(f"Wrote {records_processed} records")
        
        duration = time.time() - start_time
        
        # Complete pipeline run
        complete_pipeline_run_func(batch_id, 'SUCCESS')
        log_audit_func(batch_id, pipeline_name, 'SUCCESS',
                      rows_read=records_fetched, 
                      rows_written=records_processed,
                      runtime_seconds=duration)
        
        logger.info(f"SUCCESS - Ingested {records_processed} jobs to Bronze layer in {duration:.2f}s")
        return True, batch_id, records_processed
        
    except Exception as e:
        duration = time.time() - start_time
        error_msg = str(e)
        complete_pipeline_run_func(batch_id, 'FAILED')
        log_audit_func(batch_id, pipeline_name, 'FAILED',
                      runtime_seconds=duration, error_message=error_msg)
        logger.error(f"FAILED - {error_msg}")
        return False, batch_id, 0

In [0]:
# Log section start
log_section_start(logger, "LMIP Bronze Layer Ingestion - Arbeitnow", width=80)
logger.info(f"Started: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}")
logger.info(f"Target: {BRONZE_JOB_SNAPSHOT}")

# Execute ingestion to Bronze layer
arbeitnow_success, batch_id, records_count = ingest_to_bronze(
    source_name=SOURCE_NAME,
    fetch_function=fetch_arbeitnow_jobs,
    api_url=ARBEITNOW_API,
    extract_job_id_func=extract_arbeitnow_job_id,
    log_api_response_func=log_api_response,
    start_pipeline_run_func=start_pipeline_run,
    complete_pipeline_run_func=complete_pipeline_run,
    log_audit_func=log_audit_pipeline_run
)

# Log results
logger.info(f"Status: {'SUCCESS' if arbeitnow_success else 'FAILED'}")
if arbeitnow_success:
    log_metrics(logger, {
        "Batch ID": batch_id,
        "Records Ingested": records_count
    })
logger.info(f"Completed: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}")
log_section_end(logger, "Arbeitnow Ingestion", width=80)


In [0]:
%sql
-- View recent Arbeitnow bronze job snapshots
SELECT 
  source_name,
  batch_id,
  COUNT(*) as record_count,
  MIN(ingestion_timestamp) as first_ingested,
  MAX(ingestion_timestamp) as last_ingested
FROM bronze.bronze_job_snapshot
WHERE source_name = 'arbeitnow'
GROUP BY source_name, batch_id
ORDER BY last_ingested DESC
LIMIT 10

In [0]:
%sql
-- View Arbeitnow pipeline audit history
SELECT 
  run_id as batch_id,
  pipeline_name,
  status,
  start_time,
  end_time,
  rows_read,
  rows_written,
  ROUND(runtime_seconds, 2) as runtime_sec,
  error_message
FROM audit.audit_pipeline_runs
WHERE pipeline_name = 'bronze_ingestion_arbeitnow'
ORDER BY start_time DESC
LIMIT 20

In [0]:
%sql
-- View Arbeitnow API response telemetry
SELECT 
  source_name,
  batch_id,
  http_status_code,
  response_time_ms,
  retry_count,
  rate_limit_hit,
  logged_at
FROM bronze.bronze_api_response_log
WHERE source_name = 'arbeitnow'
ORDER BY logged_at DESC
LIMIT 20